In [ ]:
import requests
import json

# --- Configuration ---

# Replace with your actual API key
api_key = "YOUR_API_KEY"
# Replace with the specific portfolio ID you want to query
portfolio_id = "YOUR_PORTFOLIO_ID"

# --- API Call ---

# Construct the URL for the API endpoint
url = f"https://api.securityscorecard.io/portfolios/{portfolio_id}/companies"

# Define the headers required for the API request, including authorization
# Note: you can reuse this header for future API calls in this Notebook!
headers = {
    "accept": "application/json; charset=utf-8",
    # Authorization header using the specified API key format
    "Authorization": f"Token {api_key}"
}

# --- Execute Request and Process Response ---

# Send a GET request to the specified URL with the headers
response = requests.get(url, headers=headers)

# Parse the JSON response from the API
json_data = response.json()

# --- Print Specific Fields ---
for entry in json_data['entries']:
    # Extract the desired fields, using .get() for safety in case a key is missing
    domain = entry.get('domain', 'N/A')
    uuid = entry.get('uuid', 'N/A')
    name = entry.get('name', 'N/A')
    score = entry.get('score', 'N/A')
    grade = entry.get('grade', 'N/A')

    # Print the extracted information for the current entry
    print(f"Domain: {domain}")
    print(f"UUID: {uuid}")
    print(f"Name: {name}")
    print(f"Score: {score}")
    print(f"Grade: {grade}")
    print("-" * 20) # Separator for readability

In [ ]:
# This script retrieves security factor details, including issue types and counts, for each company domain obtained from the previous step.

print("--- Starting Factor Analysis (Simplified) ---")

# Check if the 'entries' key exists in the data from the first call
if 'entries' in json_data and isinstance(json_data['entries'], list):
    # Iterate through each company obtained in the first API call
    for company_entry in json_data['entries']:
        domain = company_entry.get('domain')

        # Proceed only if a valid domain was found
        if domain and domain != 'N/A':
            print(f"\n--- Fetching Factors for: {domain} ---")

            # Construct the URL for the factors endpoint using the company's domain
            factors_url = f"https://api.securityscorecard.io/companies/{domain}/factors"

            # Make the API call to get factors for the current company
            factors_response = requests.get(factors_url, headers=headers)
            # Assume the request is always successful
            factors_response.raise_for_status() # Still useful to check basic success (e.g., not 404)

            # Parse the JSON response
            factors_data = factors_response.json()

            # Check if the factors response has the expected 'entries' structure
            if 'entries' in factors_data and isinstance(factors_data['entries'], list):
                # Iterate through each factor category
                for factor_entry in factors_data['entries']:
                    factor_name = factor_entry.get('name', 'Unknown Factor')
                    # Check if the factor has an 'issue_summary' list
                    if 'issue_summary' in factor_entry and isinstance(factor_entry['issue_summary'], list):
                        print(f"  Factor: {factor_name}")
                        # Iterate through each specific issue within the factor's summary
                        for issue in factor_entry['issue_summary']:
                            issue_type = issue.get('type', 'N/A')
                            issue_count = issue.get('count', 'N/A')
                            # Print the issue type and count
                            print(f"    - Type: {issue_type}, Count: {issue_count}")
            else:
                # Basic check if the expected structure isn't found
                print(f"  Could not find 'entries' list in factors response for {domain}.")

        else:
            print("\n--- Skipping entry with missing domain ---")

else:
    print("Error: 'entries' list not found in the initial company data ('json_data'). Please run the first cell correctly.")

print("\n--- Factor Analysis Complete ---")